In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

# Define paths for each method
base_path = Path("/QRISdata/Q1851/Xiao/Working_project/benchmarking")
methods = {
    'virchow2_BLEEP': base_path / "BLEEP/Skin_data",
    'virchow2_DEEPPT': base_path / "DEEPPT/Skin_data",
    'virchow2_DeepSpace': base_path / "DeepSpace/Skin_data",
    'virchow2_STimage': base_path / "STimage/Skin_data"
}

# Collect all correlation data
all_correlations = []

for method_name, method_path in methods.items():
    cor_files = glob.glob(str(method_path / "*hvg_cor.csv"))
    
    print(f"\n{method_name}: Found {len(cor_files)} files")
    
    for cor_file in cor_files:
        try:
            df_cor = pd.read_csv(cor_file, index_col=0)
            
            if 'r' not in df_cor.columns:
                if len(df_cor.columns) == 1:
                    df_cor.columns = ['r']
                elif 'Pearson correlation' in df_cor.columns:
                    df_cor = df_cor.rename(columns={'Pearson correlation': 'r'})
            
            df_cor = df_cor.reset_index()
            
            if len(df_cor.columns) == 2:
                df_cor.columns = ['gene', 'r']
            elif 'r' in df_cor.columns:
                if 'Gene' in df_cor.columns:
                    df_cor = df_cor[['Gene', 'r']]
                    df_cor.columns = ['gene', 'r']
                elif df_cor.columns[0] != 'gene':
                    gene_col = df_cor.columns[0]
                    df_cor = df_cor[[gene_col, 'r']]
                    df_cor.columns = ['gene', 'r']
            
            df_cor['Method'] = method_name
            df_cor['File'] = Path(cor_file).stem
            
            all_correlations.append(df_cor)
            
        except Exception as e:
            print(f" ERROR loading {Path(cor_file).name}: {e}")

# Combine all data
df_all = pd.concat(all_correlations, axis=0, ignore_index=True)

print(f"Total rows (before filtering): {len(df_all)}")

# TOP 300 GENES BY MEAN PCC 
# Mean PCC for each gene across all methods 
mean_pcc_per_gene = df_all.groupby('gene')['r'].mean()

mean_abs_pcc = mean_pcc_per_gene.abs().sort_values(ascending=False)

top_n = 300
top_300_genes = mean_abs_pcc.head(top_n).index.tolist()

print(f"\nTop {top_n} genes selected by mean absolute PCC")
print(f"Mean PCC range: {mean_abs_pcc.head(top_n).min():.4f} - {mean_abs_pcc.head(top_n).max():.4f}")

# Filter data for top 300 genes
df_top300 = df_all[df_all['gene'].isin(top_300_genes)].copy()

print(f"Total rows after filtering: {len(df_top300)}")
print(f"\nRows per method after filtering:")
print(df_top300.groupby('Method').size())


print(f"\nCorrelation summary by method (top {top_n} genes by mean PCC):")
summary = df_top300.groupby('Method')['r'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(summary)

# Show top 10 genes by mean PCC
print(f"\nTop 10 genes by mean absolute PCC:")
top_10_genes_info = []
for gene in mean_abs_pcc.head(10).index:
    gene_data = df_all[df_all['gene'] == gene]
    mean_r = gene_data['r'].mean()
    mean_abs_r = gene_data['r'].abs().mean()
    top_10_genes_info.append({
        'Gene': gene,
        'Mean_PCC': mean_r,
        'Mean_Abs_PCC': mean_abs_r,
        'Count': len(gene_data)
    })
print(pd.DataFrame(top_10_genes_info))

# Create the boxplot
fig, ax = plt.subplots(figsize=(20, 6))

# Define colors
colors = ['#4472C4', '#ED7D31', '#70AD47', '#C55A5A']
method_order = ['virchow2_BLEEP', 'virchow2_DEEPPT', 'virchow2_DeepSpace', 'virchow2_STimage']

# Filter out methods with no data
available_methods = [m for m in method_order if m in df_top300['Method'].unique() and 
                     df_top300[df_top300['Method'] == m]['r'].notna().sum() > 0]
colors_filtered = [colors[method_order.index(m)] for m in available_methods]

print(f"\nAvailable methods for plotting: {available_methods}")

# Create horizontal boxplot
sns.boxplot(data=df_top300[df_top300['Method'].isin(available_methods)], 
            y='Method', x='r', 
            order=available_methods,
            palette=colors_filtered,
            hue='Method',
            legend=False,
            ax=ax,
            showfliers=True,
            flierprops=dict(marker='o', markersize=3, alpha=0.5))

pretty_names = {
    'virchow2_BLEEP': 'BLEEP',
    'virchow2_DEEPPT': 'DEEPPT',
    'virchow2_DeepSpace': 'DeepSpace',
    'virchow2_STimage': 'STimage'
}

ax.set_yticklabels([pretty_names[m] for m in available_methods])

# Customize plot
ax.set_xlabel('Pearson correlation', fontsize=45, fontweight='bold')
ax.set_ylabel('Skin Dataset', fontsize=45, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.tick_params(axis='y', labelsize=36)
ax.tick_params(axis='x', labelsize=36)

# Set x-axis limits
if len(available_methods) > 0:
    x_min = -0.2
    x_max = 1.0
    ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.savefig(f'Skin_HVG{top_n}_mean_pcc.png', dpi=600, bbox_inches='tight')
plt.savefig(f'Skin_HVG{top_n}_mean_pcc.pdf', dpi=600, bbox_inches='tight')
plt.show()

# Print detailed statistics
print(f"Detailed Statistics by Method (Top {top_n} Genes by Mean PCC):")
for method in available_methods:
    method_data = df_top300[df_top300['Method'] == method]['r']
    print(f"\n{method}:")
    print(f"  Count: {len(method_data)}")
    print(f"  Mean:   {method_data.mean():.4f}")
    print(f"  Median: {method_data.median():.4f}")
    print(f"  Std:    {method_data.std():.4f}")
    print(f"  Min:    {method_data.min():.4f}")
    print(f"  Max:    {method_data.max():.4f}")
    print(f"  Q1:     {method_data.quantile(0.25):.4f}")
    print(f"  Q3:     {method_data.quantile(0.75):.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

# Define paths for each method
base_path = Path("/QRISdata/Q1851/Xiao/Working_project/benchmarking")
methods = {
    'virchow2_BLEEP': base_path / "BLEEP/BC_data",
    'virchow2_DEEPPT': base_path / "DEEPPT/BC_data",
    'virchow2_DeepSpace': base_path / "DeepSpace/BC_data",
    'virchow2_STimage': base_path / "STimage/BC_data"
}

# Collect all correlation data
all_correlations = []

for method_name, method_path in methods.items():
    cor_files = glob.glob(str(method_path / "*hvg_cor.csv"))
    
    print(f"\n{method_name}: Found {len(cor_files)} files")
    
    for cor_file in cor_files:
        try:
            df_cor = pd.read_csv(cor_file, index_col=0)
            
            if 'r' not in df_cor.columns:
                if len(df_cor.columns) == 1:
                    df_cor.columns = ['r']
                elif 'Pearson correlation' in df_cor.columns:
                    df_cor = df_cor.rename(columns={'Pearson correlation': 'r'})
            
            df_cor = df_cor.reset_index()
            
            if len(df_cor.columns) == 2:
                df_cor.columns = ['gene', 'r']
            elif 'r' in df_cor.columns:
                if 'Gene' in df_cor.columns:
                    df_cor = df_cor[['Gene', 'r']]
                    df_cor.columns = ['gene', 'r']
                elif df_cor.columns[0] != 'gene':
                    gene_col = df_cor.columns[0]
                    df_cor = df_cor[[gene_col, 'r']]
                    df_cor.columns = ['gene', 'r']
            
            df_cor['Method'] = method_name
            df_cor['File'] = Path(cor_file).stem
            
            all_correlations.append(df_cor)
            
        except Exception as e:
            print(f" ERROR loading {Path(cor_file).name}: {e}")

# Combine all data
df_all = pd.concat(all_correlations, axis=0, ignore_index=True)

print(f"Total rows (before filtering): {len(df_all)}")

# TOP 300 GENES BY MEAN PCC 
# Mean PCC for each gene across all methods and files
mean_pcc_per_gene = df_all.groupby('gene')['r'].mean()

mean_abs_pcc = mean_pcc_per_gene.abs().sort_values(ascending=False)

top_n = 300
top_300_genes = mean_abs_pcc.head(top_n).index.tolist()

print(f"\nTop {top_n} genes selected by mean absolute PCC")
print(f"Mean PCC range: {mean_abs_pcc.head(top_n).min():.4f} - {mean_abs_pcc.head(top_n).max():.4f}")

# Filter data for top 300 genes
df_top300 = df_all[df_all['gene'].isin(top_300_genes)].copy()

print(f"Total rows after filtering: {len(df_top300)}")
print(f"\nRows per method after filtering:")
print(df_top300.groupby('Method').size())


print(f"\nCorrelation summary by method (top {top_n} genes by mean PCC):")
summary = df_top300.groupby('Method')['r'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(summary)

# Show top 10 genes by mean PCC
print(f"\nTop 10 genes by mean absolute PCC:")
top_10_genes_info = []
for gene in mean_abs_pcc.head(10).index:
    gene_data = df_all[df_all['gene'] == gene]
    mean_r = gene_data['r'].mean()
    mean_abs_r = gene_data['r'].abs().mean()
    top_10_genes_info.append({
        'Gene': gene,
        'Mean_PCC': mean_r,
        'Mean_Abs_PCC': mean_abs_r,
        'Count': len(gene_data)
    })
print(pd.DataFrame(top_10_genes_info))

# Create the boxplot
fig, ax = plt.subplots(figsize=(20, 6))

# Define colors
colors = ['#4472C4', '#ED7D31', '#70AD47', '#C55A5A']
method_order = ['virchow2_BLEEP', 'virchow2_DEEPPT', 'virchow2_DeepSpace', 'virchow2_STimage']

# Filter out methods with no data
available_methods = [m for m in method_order if m in df_top300['Method'].unique() and 
                     df_top300[df_top300['Method'] == m]['r'].notna().sum() > 0]
colors_filtered = [colors[method_order.index(m)] for m in available_methods]

print(f"\nAvailable methods for plotting: {available_methods}")

# Create horizontal boxplot
sns.boxplot(data=df_top300[df_top300['Method'].isin(available_methods)], 
            y='Method', x='r', 
            order=available_methods,
            palette=colors_filtered,
            hue='Method',
            legend=False,
            ax=ax,
            showfliers=True,
            flierprops=dict(marker='o', markersize=3, alpha=0.5))

pretty_names = {
    'virchow2_BLEEP': 'BLEEP',
    'virchow2_DEEPPT': 'DEEPPT',
    'virchow2_DeepSpace': 'DeepSpace',
    'virchow2_STimage': 'STimage'
}

ax.set_yticklabels([pretty_names[m] for m in available_methods])

# Customize plot
ax.set_xlabel('Pearson correlation', fontsize=45, fontweight='bold')
ax.set_ylabel('BC Dataset', fontsize=45, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.tick_params(axis='y', labelsize=36)
ax.tick_params(axis='x', labelsize=36)

# Set x-axis limits
if len(available_methods) > 0:
    x_min = -0.2
    x_max = 1.0
    ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.savefig(f'BC_HVG{top_n}_mean_pcc.png', dpi=600, bbox_inches='tight')
plt.savefig(f'BC_HVG{top_n}_mean_pcc.pdf', dpi=600, bbox_inches='tight')
plt.show()

# Print detailed statistics
print(f"Detailed Statistics by Method (Top {top_n} Genes by Mean PCC):")
for method in available_methods:
    method_data = df_top300[df_top300['Method'] == method]['r']
    print(f"\n{method}:")
    print(f"  Count: {len(method_data)}")
    print(f"  Mean:   {method_data.mean():.4f}")
    print(f"  Median: {method_data.median():.4f}")
    print(f"  Std:    {method_data.std():.4f}")
    print(f"  Min:    {method_data.min():.4f}")
    print(f"  Max:    {method_data.max():.4f}")
    print(f"  Q1:     {method_data.quantile(0.25):.4f}")
    print(f"  Q3:     {method_data.quantile(0.75):.4f}")